# Fichas MX/US: exposición y asignación de CFU
## tl;dr
Dos escenarios históricos normalizados por tractor equivalente; **no son bases calibradas ni importes de cotización**. IMT: 10,000 km y 25 días operativos por mes promedio. ATRI: aproximadamente 11,087.978 km y 22.333 días. No representan una comparación de costos MX/US.

## Context & Methods
Decisión: identificar qué denominadores puede estudiar el CFU sin inventar presupuesto fijo. Se reutiliza max(distancia, tiempo) del motor; no se suman dos asignaciones del mismo fijo.
### Key Assumptions
Un tractor equivalente es unidad de normalización, no flota de un tractor ni ratio de un remolque. Dividir un año entre 12 no observa estacionalidad. Días operativos no prueban días productivos del ciclo. El estrés 0.8 es analítico, no rango de benchmark. No usar estos escenarios para anualizar costos de cohortes regionales/sectoriales.
Python estándar; sin paquetes, credenciales, conexión de red, base de datos ni llamadas al motor. Ejecutar desde la carpeta de este notebook o desde la raíz del worktree. El archivo JSON contiguo conserva fuentes/localizadores y hashes de PDF.

## Data
IMT PT260 (2004), p.19 impresa / PDF 33: caso modelado T3-S2; carrocería no confirmada. [Fuente](https://imt.mx/archivos/publicaciones/publicaciontecnica/pt260.pdf). ATRI 2025, p.12 impresa / PDF 15: respondientes nacionales, operaciones 2024, no subset caja seca D2D. [Fuente](https://truckingresearch.org/wp-content/uploads/2025/07/ATRI-Operational-Costs-of-Trucking-07-2025.pdf). Existe edición 2026 pero no se obtuvieron sus tablas completas. Los escenarios se mantienen separados.

In [1]:
import json
import math
from pathlib import Path
filename = 'B2_REFERENCE_PROFILE_EXPOSURE_2026-09-29.json'
candidates = [Path(filename), Path('docs/research') / filename]
data_path = next(path for path in candidates if path.is_file())
dossier = json.loads(data_path.read_text(encoding='utf-8'))
sources = {source['id']: source for source in dossier['sources']}
results = []
for profile in dossier['profiles']:
    source = sources[profile['exposureScenarioSourceId']]
    distance = source['annualDistance']
    annual_km = distance['value'] * (dossier['method']['mileToKm'] if distance['unit'].startswith('mile/') else 1)
    days = source['annualOperatingDays']['value']
    monthly_km, monthly_days = annual_km / 12, days / 12
    results.append({'country': profile['country'], 'monthly_km': monthly_km, 'monthly_days': monthly_days, 'km_per_operating_day': annual_km / days, 'budget_fraction_per_km': 1 / monthly_km, 'budget_fraction_per_day': 1 / monthly_days})
print('Escenarios históricos separados; C = presupuesto fijo mensual por tractor equivalente, aún desconocido.')
for row in results:
    print(f"{row['country']}: {row['monthly_km']:.6f} km/mes promedio; {row['monthly_days']:.6f} días operativos/mes promedio; umbral {row['km_per_operating_day']:.6f} km/día")
    print(f"  Fracción de C: {row['budget_fraction_per_km']:.10f} por km; {row['budget_fraction_per_day']:.10f} por día; usar MAX, no SUMA.")

Escenarios históricos separados; C = presupuesto fijo mensual por tractor equivalente, aún desconocido.
MX: 10000.000000 km/mes promedio; 25.000000 días operativos/mes promedio; umbral 400.000000 km/día
  Fracción de C: 0.0001000000 por km; 0.0400000000 por día; usar MAX, no SUMA.
US: 11087.977824 km/mes promedio; 22.333333 días operativos/mes promedio; umbral 496.476619 km/día
  Fracción de C: 0.0000901878 por km; 0.0447761194 por día; usar MAX, no SUMA.


## Results
CFU candidato = C × max(km del ciclo / km del mes promedio, días del ciclo / días operativos del mes promedio). Por encima del umbral km/día domina distancia; por debajo domina tiempo. Es una regla algebraica condicionada a reconciliar días y alcance, **no un costo calculado**. Reducción uniforme del 20% en exposición, con mismo C/ciclo, multiplica asignación por 1.25.

In [2]:
assert len(results) == 2 and {row['country'] for row in results} == {'MX', 'US'}
assert math.isclose(results[0]['monthly_km'], 10000)
assert math.isclose(results[0]['monthly_days'], 25)
assert math.isclose(results[1]['monthly_km'], 11087.977824)
assert math.isclose(results[1]['monthly_days'], 268 / 12)
checks = 5
for row in results:
    assert row['monthly_km'] > 0 and 0 < row['monthly_days'] <= 31
    assert math.isclose(row['monthly_km'] / row['monthly_days'], row['km_per_operating_day'])
    # Fractions only: no money or actual trip is introduced.
    distance_fraction, time_fraction = 0.3, 0.2
    base = max(distance_fraction, time_fraction)
    stressed = max(distance_fraction / dossier['sensitivity']['exposureMultiplier'], time_fraction / dossier['sensitivity']['exposureMultiplier'])
    assert math.isclose(stressed / base, dossier['sensitivity']['allocatedCostMultiplier'])
    assert dossier['outputs']['cfuUsd'] is None and dossier['adoptedValues'] == 0
    checks += 4
print(f'{checks} comprobaciones: PASS; estrés 0.8 → multiplicador 1.25; CFU/CVU/precios permanecen null.')

13 comprobaciones: PASS; estrés 0.8 → multiplicador 1.25; CFU/CVU/precios permanecen null.


## Takeaways
Las fichas proponen tracto usado propio para estudiar capital por activo; financiamiento no determinado y remolque compartido sin responsable definido, no costo cero. Ninguna fuente observada prueba conjuntamente ese perfil actual y su exposición. No imputar FTE, flota, loaded/empty ni stock de remolques. Próximo: capital por activo, con costo capitalizable, vida restante, residual e inclusiones; conservar presupuesto fijo faltante hasta soporte. Ledger y plantilla gobernados no se modifican.